In [79]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [80]:
# Load Data
data = pd.read_csv("../data/preprocessed_data.csv")
test = pd.read_csv("../data/preprocessed_test.csv")

data.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,DeckF,DeckG,DeckT,DeckG_midCabin,DeckE_midCabin,DeckF_midCabin,CryoSleep_spending_conflict,NoSpending_notCryo,total_spending_family,avg_spending_family
0,0001_01,Europa,0,B/0/P,TRAPPIST-1e,39.0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0.0
1,0002_01,Earth,0,F/0/S,TRAPPIST-1e,24.0,0,109,9,25,...,1,0,0,0,0,0,0,0,736,736.0
2,0003_01,Europa,0,A/0/S,TRAPPIST-1e,58.0,1,43,3576,0,...,0,0,0,0,0,0,0,0,15559,7779.5
3,0003_02,Europa,0,A/0/S,TRAPPIST-1e,33.0,0,0,1283,371,...,0,0,0,0,0,0,0,0,15559,7779.5
4,0004_01,Earth,0,F/1/S,TRAPPIST-1e,16.0,0,303,70,151,...,1,0,0,0,0,0,0,0,1091,1091.0


In [81]:
# Selected Columns
selected_columns = [
    "CryoSleep", "Age", "VIP", "RoomService", "FoodCourt",
    "ShoppingMall", "Spa", "VRDeck", "Cabin_num", "Side",
    "CabinMidFlag", "HomePlanetEarth", "HomePlanetEuropa",
    "HomePlanetMars", "Destination55 Cancri e",
    "DestinationPSO J318.5-22", "DestinationTRAPPIST-1e",
    "Group1", "Group2", "Group3", "Group4", "Group5",
    "Group6", "Group7", "DeckA", "DeckB", "DeckC",
    "DeckD", "DeckE", "DeckF", "DeckG",
    "FamilyMembersCabinCount", "avg_spending_family"
]

target_column = "Transported" 

In [82]:
# Prepare Data
X = data[selected_columns]
y = data[target_column]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

In [83]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_holdout_scaled = scaler.transform(X_holdout)

In [84]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # binary output
])

d:\SBU DS\Spring 2026\AMS 580 - Statistical Learning\Spaceship titanic project\spaceship-titanic\env\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [85]:
# Prepare Data
X = data[selected_columns]
y = data[target_column]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

In [86]:
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [87]:
from sklearn.utils import class_weight
import numpy as np

# Compute class weights
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dictionary for Keras
class_weights = dict(enumerate(class_weights_array))

In [88]:
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    class_weight=class_weights
)

Epoch 1/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6625 - loss: 0.6005 - val_accuracy: 0.7629 - val_loss: 0.4856
Epoch 2/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7547 - loss: 0.4912 - val_accuracy: 0.7904 - val_loss: 0.4452
Epoch 3/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7761 - loss: 0.4583 - val_accuracy: 0.7942 - val_loss: 0.4344
Epoch 4/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7832 - loss: 0.4423 - val_accuracy: 0.8000 - val_loss: 0.4281
Epoch 5/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7929 - loss: 0.4357 - val_accuracy: 0.8038 - val_loss: 0.4257
Epoch 6/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7895 - loss: 0.4326 - val_accuracy: 0.8026 - val_loss: 0.4260
Epoch 7/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7951 - loss: 0.4265 - val_accuracy: 0.8070 - val_loss: 0.4229
Epoch 8/50
196/196 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7972 - loss: 0.4257 - val_accuracy: 0.

In [89]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_21 (Dense)                │ (None, 64)             │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,869 (50.27 KB)

 Trainable params: 4,289 (16.75 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,580 (33.52 KB)

In [90]:
# If your model output is probabilities
preds = model.predict(X_holdout)

# Convert to 0 or 1
preds_binary = (preds > 0.5).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:")
print(confusion_matrix(y_holdout, preds_binary))

print("\nClassification Report:")
print(classification_report(y_holdout, preds_binary))

28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Confusion Matrix:
[[408  24]
 [341  97]]

Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.94      0.69       432
           1       0.80      0.22      0.35       438

    accuracy                           0.58       870
   macro avg       0.67      0.58      0.52       870
weighted avg       0.67      0.58      0.52       870

